In [14]:
import os
import sys
sys.path.append(os.getcwd()+'/EasyEdit')
try:
    from EasyEdit.easyeditor import (
        FTHyperParams,
        IKEHyperParams,
        KNHyperParams,
        MEMITHyperParams,
        ROMEHyperParams,
        LoRAHyperParams,
        MENDHyperParams,
        SERACHparams,
        WISEHyperParams,
        )

    from EasyEdit.easyeditor import BaseEditor
    from EasyEdit.easyeditor.models.ike import encode_ike_facts
    from sentence_transformers import SentenceTransformer
    from EasyEdit.easyeditor import KnowEditDataset

except ImportError:
    from easyeditor import (
        FTHyperParams,
        IKEHyperParams,
        KNHyperParams,
        MEMITHyperParams,
        ROMEHyperParams,
        LoRAHyperParams,
        MENDHyperParams,
        SERACHparams,
        WISEHyperParams,
        )

    from easyeditor import BaseEditor
    from easyeditor.models.ike import encode_ike_facts
    from sentence_transformers import SentenceTransformer
    from easyeditor import KnowEditDataset
     
from typing import List   

def load_counterfact_prompt_loc(filepath: str, size: int) -> List:
    datas = KnowEditDataset(filepath, size=size)
    prompts = [data['prompt'] for data in datas]

    locality_rs = [data['locality_rs'] for data in datas]
    locality_f = [data['locality_f'] for data in datas]
    locality_Relation_Specificity_prompts = []
    locality_Relation_Specificity_ans = []
    locality_Forgetfulness_prompts = []
    locality_Forgetfulness_ans = []

    locality_data = [locality_rs, locality_f]
    locality_prompts = [locality_Relation_Specificity_prompts, locality_Forgetfulness_prompts]
    locality_answers = [locality_Relation_Specificity_ans, locality_Forgetfulness_ans]
    for data, local_prompts, local_answers in zip(locality_data, locality_prompts, locality_answers):
        for item in data:
            if item is None:
                local_prompts.append(None)
                local_answers.append(None)
            else:
                temp_prompts = []
                temp_answers = []
                for pr in item:
                    prompt = pr["prompt"]
                    an = pr["ground_truth"]
                    while isinstance(an, list):
                        an = an[0]
                    if an.strip() == "":
                        continue
                    temp_prompts.append(prompt)
                    temp_answers.append(an)
                local_prompts.append(temp_prompts)
                local_answers.append(temp_answers)
    assert len(prompts) == len(locality_Relation_Specificity_prompts) == len(locality_Forgetfulness_prompts)
    locality_inputs = {}
    portability_inputs = {}

    locality_inputs = {
        'Relation_Specificity': {
            'prompt': locality_Relation_Specificity_prompts,
            'ground_truth': locality_Relation_Specificity_ans
        },
        'Forgetfulness': {
            'prompt': locality_Forgetfulness_prompts,
            'ground_truth': locality_Forgetfulness_ans
        }
    }
     
    relation_spec_prompts = locality_inputs['Relation_Specificity']['prompt']
     
    forget_prompts = locality_inputs['Forgetfulness']['prompt']
    
    loc_prompts = [(r, f) for r, f in zip(relation_spec_prompts, forget_prompts)] 

    return [prompts, loc_prompts]
          


In [15]:
a = load_counterfact_prompt_loc(r'O:\bishe3\EasyEdit\data\KnowEdit\benchmark_wiki_counterfact_train_cf.json', 100)

In [16]:
a

[['The name of the country which Goursez Vreizh is associated with is',
  'The name of the position held by Frederic Piesch is',
  'The occupation of Martín Solares is',
  'The gender of Jallal is',
  'The gender of Jose L Castillo is',
  'The occupation of Emily I Jones is',
  'The name of the country which canton of Orcières is associated with is',
  'The occupation of G.L. Defer is',
  'The occupation of Nicholas D Rintala is',
  'The occupation of Stanislav Rössler is',
  'The name of the mother of Stephana Warnock is',
  'The occupation of Darren Finlay is',
  'The gender of Henry John Gepp is',
  "boxing at the 2010 Asian Games – men's 69 kg is followed by",
  'The name of the capital city of canton of Bagnères-de-Bigorre is',
  'The place of birth of Nicolás Méndez Casariego is',
  'The name of the position held by Thomas Phillipps Lamb is',
  'The gender of Yoshida Keigo is',
  '2041 BC follows',
  "1981 Lithuanian Badminton Championships – women's singles follows",
  'The gend

In [20]:
a[0]

['The name of the country which Goursez Vreizh is associated with is',
 'The name of the position held by Frederic Piesch is',
 'The occupation of Martín Solares is',
 'The gender of Jallal is',
 'The gender of Jose L Castillo is',
 'The occupation of Emily I Jones is',
 'The name of the country which canton of Orcières is associated with is',
 'The occupation of G.L. Defer is',
 'The occupation of Nicholas D Rintala is',
 'The occupation of Stanislav Rössler is',
 'The name of the mother of Stephana Warnock is',
 'The occupation of Darren Finlay is',
 'The gender of Henry John Gepp is',
 "boxing at the 2010 Asian Games – men's 69 kg is followed by",
 'The name of the capital city of canton of Bagnères-de-Bigorre is',
 'The place of birth of Nicolás Méndez Casariego is',
 'The name of the position held by Thomas Phillipps Lamb is',
 'The gender of Yoshida Keigo is',
 '2041 BC follows',
 "1981 Lithuanian Badminton Championships – women's singles follows",
 'The gender of Anna Sophie Gas

In [21]:
a[1]

[(['The name of the founder of Goursez Vreizh is'], None),
 (['The gender of Frederic Piesch is'],
  ['The name of the position held by Frederic Piesch, which is not Archbishop of León, Mexico, is']),
 (['The gender of Martín Solares is'],
  ['The occupation of Martín Solares, which is not geohasher, is']),
 (['The place of birth of Jallal is'], None),
 (['The occupation of Jose L Castillo is'], None),
 (['The gender of Emily I Jones is'],
  ['The occupation of Emily I Jones, which is not philatelist, is']),
 (['The name of the capital city of canton of Orcières is'], None),
 (None, ['The occupation of G.L. Defer, which is not Greek prefect, is']),
 (['The name of the employer of Nicholas D Rintala is'],
  ['The occupation of Nicholas D Rintala, which is not police dog, is']),
 (['The gender of Stanislav Rössler is'],
  ['The occupation of Stanislav Rössler, which is not bayan, is']),
 (['The name of the father of Stephana Warnock is'], None),
 (None,
  ['The occupation of Darren Finla

In [8]:
a[1].keys()

dict_keys(['Relation_Specificity', 'Forgetfulness'])

In [12]:
len(a[1]['Relation_Specificity']['prompt'])

100